# 7.8 — LeNet & AlexNet

LeNet and AlexNet are the historical templates for modern convolutional vision models: keep images spatial for as long as local evidence matters, repeatedly apply shared filters and shrinking operations, then flatten only when a classifier has enough high-level features to decide. In this notebook we rebuild their core arithmetic from scratch with NumPy so every output shape, activation, parameter count, and regularization choice is inspectable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build the architecture ideas one at a time. Run each cell in order and read the printed intermediate values — convolution, pooling, shape accounting, ReLU, flattening, and parameter budgets are all derived in small NumPy pieces. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + handmade CNN arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy images and filters.

### 1. Convolution keeps local structure instead of flattening early

A convolutional layer learns a small local template, slides it over the image, and reuses the same weights at every location. That is the key difference from flattening: neighboring pixels stay neighbors, so a stroke detector can fire wherever the stroke appears.

In [ ]:
img_w = np.zeros((8, 8))                         # a tiny grayscale image.
img_w[2:6, 3:5] = 1.0                            # draw a vertical bright stroke.
edge_w = np.array([[-1., 0., 1.],                # a vertical-edge filter.
                   [-1., 0., 1.],
                   [-1., 0., 1.]])
print("image shape:", img_w.shape, "filter shape:", edge_w.shape)
print("filter weights:\n", edge_w)

▶ What you'll see: an 8×8 image and one 3×3 filter; the filter is tiny compared with the image.

In [ ]:
out_w = np.zeros((6, 6))                          # valid convolution output: 8 - 3 + 1 = 6.
for r_w in range(out_w.shape[0]):                 # slide over output rows.
    for c_w in range(out_w.shape[1]):             # slide over output columns.
        patch_w = img_w[r_w:r_w+3, c_w:c_w+3]     # local 3×3 neighborhood.
        out_w[r_w, c_w] = np.sum(patch_w * edge_w) # template match score.
print("conv output shape:", out_w.shape)
print("max response:", out_w.max(), "min response:", out_w.min())
assert out_w.shape == (6, 6)

▶ What you'll see: a 6×6 response map where large positive/negative values sit near the stroke boundaries.

In [ ]:
fig_w, ax_w = plt.subplots(1, 3, figsize=(8, 2.6))
ax_w[0].imshow(img_w, cmap="gray"); ax_w[0].set_title("image")
ax_w[1].imshow(edge_w, cmap="coolwarm"); ax_w[1].set_title("3×3 filter")
ax_w[2].imshow(out_w, cmap="coolwarm"); ax_w[2].set_title("response map")
for a_w in ax_w: a_w.set_xticks([]); a_w.set_yticks([])
plt.suptitle("1: one shared local filter scans the image"); plt.show()

▶ What you'll see: the response map lights up at positions where the local 3×3 patch matches the edge pattern.

*Why it's done this way:* the same 3×3 weights are reused at every spatial position, so the model learns one detector and applies it everywhere. Mathematically, each output is a dot product between the same kernel and a different local patch; the weight sharing gives translation sensitivity without creating a separate dense weight for every pixel location.

### 2. The CNN shape equation is architecture bookkeeping

LeNet and AlexNet diagrams are mostly the same formula repeated: for input size $H$, padding $p$, kernel $k$, and stride $s$, the output height is $\left\lfloor (H+2p-k)/s \right\rfloor+1$. The floor appears because only complete kernel landings count.

In [ ]:
def conv_shape_w(H_w, W_w, k_w, stride_w=1, pad_w=0):
    h_w = (H_w + 2 * pad_w - k_w) // stride_w + 1
    w_w = (W_w + 2 * pad_w - k_w) // stride_w + 1
    return h_w, w_w

lenet_h_w, lenet_w_w = conv_shape_w(32, 32, 5, stride_w=1, pad_w=0)
alex_h_w, alex_w_w = conv_shape_w(227, 227, 11, stride_w=4, pad_w=0)
print("LeNet first conv spatial:", (lenet_h_w, lenet_w_w))
print("AlexNet first conv spatial:", (alex_h_w, alex_w_w))
assert (lenet_h_w, lenet_w_w) == (28, 28)
assert (alex_h_w, alex_w_w) == (55, 55)

▶ What you'll see: LeNet maps 32×32 to 28×28, while AlexNet maps 227×227 to 55×55 immediately.

In [ ]:
landings_w = []
for start_w in range(0, 227 - 11 + 1, 4):        # AlexNet's stride-4 kernel starts.
    landings_w.append(start_w)
print("first five starts:", landings_w[:5])
print("last start:", landings_w[-1], "number of starts:", len(landings_w))
assert len(landings_w) == 55

▶ What you'll see: the first kernel starts at 0, then 4, 8, 12, ... and there are exactly 55 valid landings.

In [ ]:
plt.figure(figsize=(5, 2.8))
plt.scatter(landings_w, np.zeros_like(landings_w), s=15, color="teal")
plt.title("2: stride-4 valid kernel landings along one axis")
plt.xlabel("input coordinate of 11×11 kernel start"); plt.yticks([])
plt.show()

▶ What you'll see: AlexNet samples a coarse grid of starts, saving compute but skipping many possible pixel offsets.

*Why it's done this way:* output size is not a convention; it is the count of legal positions where the kernel fits. Stride reduces that count by jumping over starts, and padding increases it by giving the kernel extra border area. Architecture design is therefore a tradeoff between retaining spatial detail and reducing the cost of later layers.

### 3. Pooling shrinks maps without making new channels

Pooling summarizes each channel locally. A 2×2 max pool with stride 2 halves the height and width, but it does not learn new filters and it does not change the channel count; it just keeps the strongest evidence in each small neighborhood.

In [ ]:
feat_w = np.array([[0.1, 0.2, 0.4, 0.3],
                   [0.0, 1.5, 0.2, 0.1],
                   [0.3, 0.4, 2.0, 0.2],
                   [0.2, 0.1, 0.5, 0.6]])
print("feature map shape:", feat_w.shape)
print(feat_w)

▶ What you'll see: a 4×4 feature map with two strong activations that a pool should preserve.

In [ ]:
pool_w = np.zeros((2, 2))
for r_w in range(2):
    for c_w in range(2):
        block_w = feat_w[2*r_w:2*r_w+2, 2*c_w:2*c_w+2]
        pool_w[r_w, c_w] = np.max(block_w)
print("pooled map:\n", pool_w)
assert np.allclose(pool_w, [[1.5, 0.4], [0.4, 2.0]])

▶ What you'll see: each 2×2 block becomes one number — the maximum activation in that local region.

In [ ]:
fig_w, ax_w = plt.subplots(1, 2, figsize=(6, 2.8))
ax_w[0].imshow(feat_w, cmap="viridis"); ax_w[0].set_title("before pool 4×4")
ax_w[1].imshow(pool_w, cmap="viridis"); ax_w[1].set_title("after pool 2×2")
for a_w in ax_w: a_w.set_xticks([]); a_w.set_yticks([])
plt.suptitle("3: max pooling keeps local winners"); plt.show()

▶ What you'll see: the grid becomes smaller while the strongest responses remain visible.

*Why it's done this way:* pooling is a local invariance step. If a stroke shifts by one pixel inside a 2×2 block, the maximum can stay the same, so later layers see more stable evidence. Because the operation is applied independently per channel, it summarizes locations, not feature types.

### 4. LeNet is a small spatial pipeline before a classifier

LeNet's classic digit pipeline repeats convolution and pooling until the maps are small enough to flatten. The important habit is to track height, width, channels, and parameters at each step instead of treating the architecture as a drawing.

In [ ]:
H_w, W_w, C_w = 32, 32, 1                         # LeNet input: padded digit.
C1_w = (conv_shape_w(H_w, W_w, 5, 1, 0), 6)       # 5×5 conv, 6 filters.
S2_w = (conv_shape_w(C1_w[0][0], C1_w[0][1], 2, 2, 0), 6) # 2×2 pool.
C3_w = (conv_shape_w(S2_w[0][0], S2_w[0][1], 5, 1, 0), 16) # next conv.
S4_w = (conv_shape_w(C3_w[0][0], C3_w[0][1], 2, 2, 0), 16) # next pool.
print("C1:", C1_w, "S2:", S2_w, "C3:", C3_w, "S4:", S4_w)
assert C1_w == ((28, 28), 6)
assert S2_w == ((14, 14), 6)
assert C3_w == ((10, 10), 16)
assert S4_w == ((5, 5), 16)

▶ What you'll see: the spatial grid shrinks from 32×32 to 5×5 while channels rise from 1 to 16.

In [ ]:
lenet_c1_params_w = 5 * 5 * 1 * 6 + 6             # weights plus one bias per filter.
lenet_c3_params_w = 5 * 5 * 6 * 16 + 16           # if fully connected across previous channels.
flatten_w = 5 * 5 * 16
print("C1 params:", lenet_c1_params_w)
print("C3 params:", lenet_c3_params_w)
print("flattened features before dense:", flatten_w)
assert lenet_c1_params_w == 156
assert flatten_w == 400

▶ What you'll see: the early convolution has only 156 parameters, and the classifier receives 400 spatial features.

In [ ]:
stages_w = ["input", "C1", "S2", "C3", "S4"]
areas_w = [32*32, 28*28, 14*14, 10*10, 5*5]
channels_w = [1, 6, 6, 16, 16]
plt.figure(figsize=(6, 3))
plt.plot(stages_w, areas_w, marker="o", label="spatial area")
plt.plot(stages_w, channels_w, marker="s", label="channels")
plt.title("4: LeNet shrinks space while adding features")
plt.legend(); plt.show()

▶ What you'll see: spatial area drops quickly, while the number of feature maps grows more gently.

*Why it's done this way:* early layers must preserve layout because strokes and corners are local. Pooling reduces the grid only after filters have detected useful evidence. Flattening is delayed until 5×5×16 is compact enough that a dense classifier can combine high-level features without drowning in pixel-level parameters.

### 5. AlexNet scales the same idea with coarse first strides and ReLU

AlexNet keeps the convolution → nonlinearity → pooling pattern but uses a much larger RGB input, many more filters, and ReLU activations. The first 11×11 stride-4 layer is deliberately coarse: it reduces ImageNet-sized inputs before later layers become too expensive.

In [ ]:
alex_out_w = conv_shape_w(227, 227, 11, stride_w=4, pad_w=0)
alex_params_w = 11 * 11 * 3 * 96 + 96
print("AlexNet conv1 output:", alex_out_w, "channels: 96")
print("AlexNet conv1 params:", alex_params_w)
assert alex_out_w == (55, 55)
assert alex_params_w == 34944

▶ What you'll see: the first layer produces 55×55×96 activations and already has 34,944 parameters including biases.

In [ ]:
z_w = np.linspace(-4, 4, 17)
relu_w = np.maximum(0, z_w)
print("z:", z_w[:5], "...", z_w[-5:])
print("ReLU(z):", relu_w[:5], "...", relu_w[-5:])
assert np.all(relu_w[z_w < 0] == 0)

▶ What you'll see: negative pre-activations become exactly 0, while positive values pass through unchanged.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(z_w, relu_w, marker="o", label="ReLU")
plt.plot(z_w, 1/(1+np.exp(-z_w)), label="sigmoid", linestyle="--")
plt.title("5: ReLU does not saturate on the positive side")
plt.xlabel("pre-activation"); plt.ylabel("activation"); plt.legend(); plt.show()

▶ What you'll see: ReLU is flat for negatives and linear for positives, unlike sigmoid's bounded S-shape.

*Why it's done this way:* a large natural-image model needs fast optimization. ReLU gives a derivative of 1 for positive inputs, so strong positive evidence keeps passing gradients backward instead of saturating. The coarse first stride makes the computation manageable, but it also explains the pitfall: small details can be skipped before the network has a chance to analyze them.

### 6. Fully connected layers can dominate the parameter budget

Convolutional layers share weights spatially, but a dense layer connects every flattened activation to every output unit. AlexNet's late classifier therefore contains tens of millions of parameters even after pooling has shrunk the maps.

In [ ]:
late_h_w, late_w_w, late_c_w = 6, 6, 256
fc_units_w = 4096
flat_w = late_h_w * late_w_w * late_c_w
fc_params_w = flat_w * fc_units_w + fc_units_w
print("flattened inputs:", flat_w)
print("dense parameters:", fc_params_w)
assert flat_w == 9216
assert fc_params_w == 37752832

▶ What you'll see: a 6×6×256 tensor becomes 9,216 inputs, and the 4,096-unit dense layer has 37,752,832 parameters.

In [ ]:
conv5_params_w = 3 * 3 * 256 * 256 + 256
ratio_w = fc_params_w / conv5_params_w
print("example 3×3 conv params:", conv5_params_w)
print("dense / conv parameter ratio:", round(ratio_w, 1))
assert conv5_params_w == 590080

▶ What you'll see: one dense layer can have more than 60 times the parameters of a large 3×3 convolution.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["3×3 conv", "dense 9216→4096"], [conv5_params_w, fc_params_w], color=["teal", "crimson"])
plt.yscale("log")
plt.title("6: dense classifiers can dominate memory")
plt.ylabel("parameters (log scale)"); plt.show()

▶ What you'll see: the dense bar towers over the convolution bar even on a log scale.

*Why it's done this way:* flattening discards spatial sharing. Once each activation is connected separately to each dense unit, the parameter count multiplies by input features times output units. AlexNet needed dropout and data augmentation partly because this huge classifier could memorize training images unless regularized.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, loops, matrix math, and tiny CNN operations.
import matplotlib.pyplot as plt # load Matplotlib so filters, feature maps, and curves can be inspected visually.
np.random.seed(0) # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Make a tiny image and filter

**Goal.** Create the smallest ingredients of a convolution, because LeNet and AlexNet both begin by matching local filters against image patches. We build it in 2 steps.

In [ ]:
img_b1 = np.zeros((6, 6)) # Create a tiny grayscale image.
img_b1[2:4, 1:5] = 1.0 # Add a horizontal bright stroke.
filt_b1 = np.array([[1., 1., 1.], [0., 0., 0.], [-1., -1., -1.]]) # Define a horizontal-edge detector.
print("image shape:", img_b1.shape, "filter shape:", filt_b1.shape) # Inspect the two objects before convolving.

▶ What you'll see: a 6×6 image and a 3×3 filter that responds to horizontal changes.

In [ ]:
plt.figure(figsize=(5, 2.4)) # Create a compact side-by-side visualization.
plt.subplot(1, 2, 1); plt.imshow(img_b1, cmap="gray"); plt.title("image"); plt.xticks([]); plt.yticks([]) # Show the toy image.
plt.subplot(1, 2, 2); plt.imshow(filt_b1, cmap="coolwarm"); plt.title("filter"); plt.xticks([]); plt.yticks([]) # Show positive and negative filter weights.
plt.show() # Display the figure.

▶ What you'll see: the image contains one bright bar, and the filter has positive weights above negative weights.

👀 Takeaway: a CNN filter is a small learnable template that will be reused across the image.

### Basic 2 — Slide one filter by hand

**Goal.** Compute one valid convolution response map, because every CNN feature map is a grid of local dot products. We build it in 3 steps.

In [ ]:
img_b2 = np.zeros((6, 6)) # Recreate the toy image for this example.
img_b2[2:4, 1:5] = 1.0 # Add the same horizontal stroke.
filt_b2 = np.array([[1., 1., 1.], [0., 0., 0.], [-1., -1., -1.]]) # Recreate the horizontal-edge filter.
out_b2 = np.zeros((4, 4)) # Valid 3×3 convolution on 6×6 has 6 - 3 + 1 = 4 rows and columns.
print("output initialized with shape:", out_b2.shape) # Inspect the shape before filling it.

▶ What you'll see: the output grid is 4×4 because only complete 3×3 landings are allowed.

In [ ]:
for r_b2 in range(4): # Loop over valid top-left patch rows.
    for c_b2 in range(4): # Loop over valid top-left patch columns.
        patch_b2 = img_b2[r_b2:r_b2+3, c_b2:c_b2+3] # Extract one local patch.
        out_b2[r_b2, c_b2] = np.sum(patch_b2 * filt_b2) # Store the patch-filter dot product.
print("response map:\n", out_b2) # Inspect every local response.
assert out_b2.shape == (4, 4) # Verify the valid-convolution output size.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a heatmap for the response map.
plt.imshow(out_b2, cmap="coolwarm") # Draw negative and positive responses with different colors.
plt.colorbar(label="filter response") # Add a scale for the dot-product values.
plt.title("Basic 2: handmade convolution") # Title the plot.
plt.show() # Display the map.

▶ What you'll see: strong responses appear where the filter overlaps the edge of the bright bar.

👀 Takeaway: convolution is repeated local dot products with shared weights.

### Basic 3 — Use the output-size formula

**Goal.** Predict convolution dimensions before running code, because architecture design is shape accounting repeated layer by layer. We build it in 2 steps.

In [ ]:
H_b3, W_b3, k_b3, s_b3, p_b3 = 32, 32, 5, 1, 0 # Set LeNet's first convolution hyperparameters.
hout_b3 = (H_b3 + 2 * p_b3 - k_b3) // s_b3 + 1 # Apply the CNN height formula.
wout_b3 = (W_b3 + 2 * p_b3 - k_b3) // s_b3 + 1 # Apply the CNN width formula.
print("LeNet C1 spatial size:", (hout_b3, wout_b3)) # Inspect the computed spatial output.
assert (hout_b3, wout_b3) == (28, 28) # Verify the classic LeNet number.

▶ What you'll see: a 32×32 digit becomes a 28×28 feature map after a 5×5 valid convolution.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact comparison chart.
plt.bar(["input H", "output H"], [H_b3, hout_b3], color=["gray", "teal"]) # Compare height before and after the convolution.
plt.title("Basic 3: valid convolution shrinks the grid") # Title the shape plot.
plt.ylabel("pixels") # Label the scale.
plt.show() # Display the bar chart.

▶ What you'll see: the output height is smaller because border positions cannot center a full 5×5 kernel without padding.

👀 Takeaway: the shape formula counts legal kernel landing positions.

### Basic 4 — Count convolution parameters

**Goal.** Count weights and biases in LeNet's first layer, because CNN efficiency comes from sharing a small kernel across many positions. We build it in 2 steps.

In [ ]:
kernel_b4, in_ch_b4, out_ch_b4 = 5, 1, 6 # Set LeNet C1 kernel size, input channels, and filters.
weights_b4 = kernel_b4 * kernel_b4 * in_ch_b4 * out_ch_b4 # Count one 5×5×1 kernel per output channel.
biases_b4 = out_ch_b4 # Count one bias per filter.
params_b4 = weights_b4 + biases_b4 # Total trainable parameters in the layer.
print("weights:", weights_b4, "biases:", biases_b4, "total:", params_b4) # Inspect the parameter count.
assert params_b4 == 156 # Verify 5*5*1*6 + 6.

▶ What you'll see: the layer has only 156 trainable parameters even though it produces 28×28×6 activations.

In [ ]:
activations_b4 = 28 * 28 * out_ch_b4 # Count output activations from the previous shape result.
plt.figure(figsize=(4.5, 3)) # Create a parameter-versus-activation comparison.
plt.bar(["parameters", "activations"], [params_b4, activations_b4], color=["orange", "teal"]) # Show how few weights create many responses.
plt.title("Basic 4: shared filters create many activations") # Title the plot.
plt.show() # Display the comparison.

▶ What you'll see: many output values are produced from a tiny set of shared weights.

👀 Takeaway: convolution separates number of learned weights from number of spatial locations.

### Basic 5 — Max-pool a feature map

**Goal.** Shrink a feature map with 2×2 max pooling, because LeNet uses pooling to reduce spatial size while keeping strong local evidence. We build it in 3 steps.

In [ ]:
feat_b5 = np.array([[0.1, 1.2, 0.2, 0.3], [0.4, 0.8, 1.5, 0.1], [0.2, 0.3, 0.4, 2.0], [0.0, 0.5, 0.2, 0.6]]) # Define one feature map.
pooled_b5 = np.zeros((2, 2)) # Prepare the 2×2 pooled output.
print("feature map shape:", feat_b5.shape) # Inspect the pre-pool size.

▶ What you'll see: a 4×4 map that will become 2×2 after stride-2 pooling.

In [ ]:
for r_b5 in range(2): # Loop over pooled rows.
    for c_b5 in range(2): # Loop over pooled columns.
        block_b5 = feat_b5[2*r_b5:2*r_b5+2, 2*c_b5:2*c_b5+2] # Extract a non-overlapping 2×2 block.
        pooled_b5[r_b5, c_b5] = np.max(block_b5) # Keep the strongest activation in the block.
print("pooled map:\n", pooled_b5) # Inspect the max-pooled result.
assert pooled_b5.shape == (2, 2) # Verify spatial halving.

In [ ]:
plt.figure(figsize=(5, 2.5)) # Create a before-after visualization.
plt.subplot(1, 2, 1); plt.imshow(feat_b5, cmap="viridis"); plt.title("before") # Show the original map.
plt.subplot(1, 2, 2); plt.imshow(pooled_b5, cmap="viridis"); plt.title("after") # Show the pooled map.
plt.show() # Display the figure.

▶ What you'll see: each 2×2 region becomes its brightest value, so the map is smaller but still highlights strong evidence.

👀 Takeaway: pooling reduces height and width, not the number of channels.

### Basic 6 — Apply ReLU to pre-activations

**Goal.** Use ReLU as AlexNet's activation, because it passes positive evidence without sigmoid-style saturation. We build it in 2 steps.

In [ ]:
z_b6 = np.array([-3.0, -0.5, 0.0, 0.5, 4.0]) # Define example pre-activation values.
relu_b6 = np.maximum(0, z_b6) # Apply ReLU elementwise.
print("z:", z_b6) # Inspect inputs.
print("ReLU(z):", relu_b6) # Inspect outputs.
assert np.allclose(relu_b6, [0.0, 0.0, 0.0, 0.5, 4.0]) # Verify the worked values.

▶ What you'll see: negatives are clipped to 0, while positive values are unchanged.

In [ ]:
x_b6 = np.linspace(-4, 4, 100) # Create a smooth input grid.
plt.figure(figsize=(4, 3)) # Create the activation plot.
plt.plot(x_b6, np.maximum(0, x_b6), color="teal") # Plot ReLU.
plt.title("Basic 6: ReLU activation") # Title the curve.
plt.xlabel("z"); plt.ylabel("max(0, z)") # Label axes.
plt.show() # Display the plot.

▶ What you'll see: the graph is flat at zero for negative inputs and linear for positive inputs.

👀 Takeaway: ReLU makes the network nonlinear while preserving positive signal strength.

### Basic 7 — Track a LeNet pipeline

**Goal.** Follow LeNet's spatial sizes through convolution and pooling, because the classifier's input size depends on every earlier layer. We build it in 2 steps.

In [ ]:
sizes_b7 = [(32, 32, 1)] # Start with a 32×32×1 padded digit.
sizes_b7.append((28, 28, 6)) # C1: 5×5 valid conv with 6 filters.
sizes_b7.append((14, 14, 6)) # S2: 2×2 pooling halves spatial size.
sizes_b7.append((10, 10, 16)) # C3: another 5×5 valid conv.
sizes_b7.append((5, 5, 16)) # S4: another 2×2 pooling.
print("LeNet sizes:", sizes_b7) # Inspect each stage.
assert sizes_b7[-1] == (5, 5, 16) # Verify the final spatial tensor before flattening.

▶ What you'll see: LeNet preserves spatial structure for several layers before reaching a 5×5×16 tensor.

In [ ]:
areas_b7 = [h_b7 * w_b7 for h_b7, w_b7, c_b7 in sizes_b7] # Compute spatial area at each stage.
channels_b7 = [c_b7 for h_b7, w_b7, c_b7 in sizes_b7] # Extract channel counts.
plt.figure(figsize=(5, 3)) # Create a compact stage plot.
plt.plot(areas_b7, marker="o", label="spatial area") # Plot area shrinkage.
plt.plot(channels_b7, marker="s", label="channels") # Plot channel growth.
plt.title("Basic 7: LeNet shape trajectory") # Title the plot.
plt.legend(); plt.show() # Display the stage curves.

▶ What you'll see: the spatial grid shrinks while the channel count increases.

👀 Takeaway: CNNs trade spatial resolution for richer learned feature channels.

### Basic 8 — Compute AlexNet's first layer shape

**Goal.** Reproduce AlexNet conv1's 55×55×96 output, because large stride is how the early model made ImageNet images manageable. We build it in 2 steps.

In [ ]:
H_b8, W_b8, k_b8, s_b8, p_b8, filters_b8 = 227, 227, 11, 4, 0, 96 # Set AlexNet conv1 hyperparameters.
hout_b8 = (H_b8 + 2 * p_b8 - k_b8) // s_b8 + 1 # Count vertical kernel landings.
wout_b8 = (W_b8 + 2 * p_b8 - k_b8) // s_b8 + 1 # Count horizontal kernel landings.
print("AlexNet conv1 output:", (hout_b8, wout_b8, filters_b8)) # Inspect the full output tensor shape.
assert (hout_b8, wout_b8, filters_b8) == (55, 55, 96) # Verify the canonical layer shape.

▶ What you'll see: the first AlexNet layer reduces 227×227 RGB input to a 55×55 grid with 96 channels.

In [ ]:
starts_b8 = np.arange(0, H_b8 - k_b8 + 1, s_b8) # List valid stride-4 kernel starts along one axis.
plt.figure(figsize=(5, 2.6)) # Create a landing-position plot.
plt.scatter(starts_b8, np.zeros_like(starts_b8), s=12, color="purple") # Show where the kernel starts.
plt.title("Basic 8: stride-4 kernel starts") # Title the plot.
plt.xlabel("input coordinate"); plt.yticks([]) # Label the coordinate axis.
plt.show() # Display the landing grid.

▶ What you'll see: the first layer samples every fourth pixel position, not every possible position.

👀 Takeaway: stride is a compute-saving choice that also reduces fine spatial coverage.

### Basic 9 — Count AlexNet conv1 parameters

**Goal.** Count the first AlexNet layer's weights, because RGB inputs and many filters make it much larger than LeNet C1. We build it in 2 steps.

In [ ]:
k_b9, in_ch_b9, out_ch_b9 = 11, 3, 96 # Set kernel size, RGB input channels, and filters.
weights_b9 = k_b9 * k_b9 * in_ch_b9 * out_ch_b9 # Count all filter weights.
biases_b9 = out_ch_b9 # Count one bias per filter.
params_b9 = weights_b9 + biases_b9 # Total parameters.
print("AlexNet conv1 weights:", weights_b9, "biases:", biases_b9, "total:", params_b9) # Inspect the count.
assert weights_b9 == 34848 # Verify the weight count cited in the lesson content.
assert params_b9 == 34944 # Verify weights plus biases.

▶ What you'll see: 34,848 weights, or 34,944 trainable parameters with biases.

In [ ]:
lenet_weights_b9 = 5 * 5 * 1 * 6 # Count LeNet C1 weights for comparison.
plt.figure(figsize=(4.5, 3)) # Create a comparison bar chart.
plt.bar(["LeNet C1", "AlexNet conv1"], [lenet_weights_b9, weights_b9], color=["teal", "crimson"]) # Compare first-layer weights.
plt.yscale("log") # Use log scale so both bars are readable.
plt.title("Basic 9: first-layer weight counts") # Title the plot.
plt.show() # Display the comparison.

▶ What you'll see: AlexNet's first convolution has vastly more weights than LeNet's first convolution.

👀 Takeaway: input scale, color channels, and filter count all change the training problem.

### Basic 10 — Flatten before a dense classifier

**Goal.** Convert a spatial tensor into a vector and count dense parameters, because fully connected layers stop using convolutional weight sharing. We build it in 3 steps.

In [ ]:
tensor_b10 = np.arange(2 * 2 * 3).reshape(2, 2, 3) # Create a tiny spatial tensor with 3 channels.
flat_b10 = tensor_b10.reshape(-1) # Flatten all spatial and channel positions into one vector.
print("tensor shape:", tensor_b10.shape, "flat length:", flat_b10.size) # Inspect the flattening step.
assert flat_b10.size == 12 # Verify 2*2*3 features.

▶ What you'll see: a 2×2×3 feature tensor becomes a 12-number vector.

In [ ]:
late_flat_b10 = 6 * 6 * 256 # Count AlexNet-style late features.
units_b10 = 4096 # Set dense classifier width.
params_b10 = late_flat_b10 * units_b10 + units_b10 # Count dense weights plus biases.
print("dense params:", params_b10) # Inspect the parameter explosion.
assert params_b10 == 37752832 # Verify the lesson's dense-layer count.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a compact parameter plot.
plt.bar(["flattened inputs", "dense params"], [late_flat_b10, params_b10], color=["gray", "red"]) # Compare input length to parameter count.
plt.yscale("log") # Use log scale because the dense parameter count is huge.
plt.title("Basic 10: flattening enables dense connections") # Title the plot.
plt.show() # Display the chart.

▶ What you'll see: a modest 9,216-vector feeding 4,096 units creates 37.75 million parameters.

👀 Takeaway: dense classifiers are powerful but can dominate memory and overfitting risk.

## 🟡 Easy

### Easy 1 — Build a reusable multi-filter convolution

**Goal.** Implement a small multi-filter convolution, because LeNet's first layer applies several learned stroke detectors to the same image. We build it in 4 steps.

In [ ]:
img_e1 = np.zeros((8, 8, 1)) # Create an 8×8 single-channel image tensor.
img_e1[2:6, 3:5, 0] = 1.0 # Draw a vertical stroke in the only channel.
filters_e1 = np.zeros((2, 3, 3, 1)) # Prepare two 3×3×1 filters.
filters_e1[0, :, :, 0] = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]]) # First filter detects vertical edges.
filters_e1[1, :, :, 0] = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]]) # Second filter detects horizontal edges.
print("image:", img_e1.shape, "filters:", filters_e1.shape) # Inspect input and filter-bank shapes.

▶ What you'll see: one image and two filters; the output will have two channels.

In [ ]:
H_e1, W_e1, C_e1 = img_e1.shape # Read input shape.
F_e1, K_e1, _, _ = filters_e1.shape # Read filter count and kernel size.
out_e1 = np.zeros((H_e1 - K_e1 + 1, W_e1 - K_e1 + 1, F_e1)) # Allocate valid-convolution output.
for f_e1 in range(F_e1): # Loop over output channels.
    for r_e1 in range(out_e1.shape[0]): # Loop over output rows.
        for c_e1 in range(out_e1.shape[1]): # Loop over output columns.
            patch_e1 = img_e1[r_e1:r_e1+K_e1, c_e1:c_e1+K_e1, :] # Extract one local patch.
            out_e1[r_e1, c_e1, f_e1] = np.sum(patch_e1 * filters_e1[f_e1]) # Apply the current filter.
print("output shape:", out_e1.shape) # Inspect the multi-channel output.
assert out_e1.shape == (6, 6, 2) # Verify valid 3×3 convolution with two filters.

In [ ]:
print("channel maxima:", np.max(out_e1[:, :, 0]), np.max(out_e1[:, :, 1])) # Inspect which detector fires strongly.

In [ ]:
fig_e1, ax_e1 = plt.subplots(1, 2, figsize=(6, 2.8)) # Create one panel per output channel.
ax_e1[0].imshow(out_e1[:, :, 0], cmap="coolwarm"); ax_e1[0].set_title("vertical-edge channel") # Show filter 0 responses.
ax_e1[1].imshow(out_e1[:, :, 1], cmap="coolwarm"); ax_e1[1].set_title("horizontal-edge channel") # Show filter 1 responses.
for a_e1 in ax_e1: a_e1.set_xticks([]); a_e1.set_yticks([]) # Hide ticks.
plt.show() # Display the feature maps.

▶ What you'll see: the vertical-edge channel responds more clearly to the vertical stroke than the horizontal-edge channel.

👀 Takeaway: multiple filters turn one image into multiple feature maps, one learned detector per output channel.

### Easy 2 — Run a mini LeNet block

**Goal.** Combine convolution, ReLU, and pooling, because historical CNNs are engineered pipelines rather than isolated formulas. We build it in 4 steps.

In [ ]:
img_e2 = np.zeros((8, 8)) # Create a small grayscale input.
img_e2[1:7, 2:4] = 1.0 # Draw a vertical stroke.
filt_e2 = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]) # Use a vertical-edge filter.
conv_e2 = np.zeros((6, 6)) # Allocate valid convolution output.
print("input shape:", img_e2.shape) # Inspect the input size.

▶ What you'll see: the block starts from an 8×8 input.

In [ ]:
for r_e2 in range(6): # Loop over valid output rows.
    for c_e2 in range(6): # Loop over valid output columns.
        conv_e2[r_e2, c_e2] = np.sum(img_e2[r_e2:r_e2+3, c_e2:c_e2+3] * filt_e2) # Compute one filter response.
relu_e2 = np.maximum(0, conv_e2) # Apply ReLU so only positive evidence passes forward.
print("conv min/max:", conv_e2.min(), conv_e2.max()) # Inspect pre-activation range.
print("ReLU nonzero count:", int(np.sum(relu_e2 > 0))) # Inspect how many positions survive.

In [ ]:
pool_e2 = np.zeros((3, 3)) # Allocate stride-2 max-pool output from 6×6 to 3×3.
for r_e2 in range(3): # Loop over pooled rows.
    for c_e2 in range(3): # Loop over pooled columns.
        pool_e2[r_e2, c_e2] = np.max(relu_e2[2*r_e2:2*r_e2+2, 2*c_e2:2*c_e2+2]) # Max over a 2×2 block.
print("pooled shape:", pool_e2.shape) # Inspect the final block output.
assert pool_e2.shape == (3, 3) # Verify spatial reduction.

In [ ]:
fig_e2, ax_e2 = plt.subplots(1, 3, figsize=(8, 2.6)) # Create panels for each block stage.
ax_e2[0].imshow(conv_e2, cmap="coolwarm"); ax_e2[0].set_title("conv") # Show raw filter responses.
ax_e2[1].imshow(relu_e2, cmap="viridis"); ax_e2[1].set_title("ReLU") # Show positive evidence only.
ax_e2[2].imshow(pool_e2, cmap="viridis"); ax_e2[2].set_title("pool") # Show pooled summary.
for a_e2 in ax_e2: a_e2.set_xticks([]); a_e2.set_yticks([]) # Hide ticks.
plt.show() # Display the block.

▶ What you'll see: convolution creates signed evidence, ReLU drops negative evidence, and pooling compresses the positive map.

👀 Takeaway: a CNN block detects, gates, and summarizes local visual evidence.

### Easy 3 — Compare LeNet and AlexNet first layers

**Goal.** Put the first-layer arithmetic side by side, because the two architectures differ in input scale, channel count, stride, and filter count. We build it in 3 steps.

In [ ]:
names_e3 = ["LeNet C1", "AlexNet conv1"] # Label the two historical first layers.
inputs_e3 = np.array([[32, 32, 1], [227, 227, 3]]) # Store H, W, C for each architecture.
kernels_e3 = np.array([5, 11]) # Store kernel sizes.
strides_e3 = np.array([1, 4]) # Store strides.
filters_e3 = np.array([6, 96]) # Store output channel counts.
print("inputs HWC:\n", inputs_e3) # Inspect the comparison table.

▶ What you'll see: AlexNet starts from a much larger RGB image and uses many more filters.

In [ ]:
spatial_e3 = [] # Store output spatial sizes.
params_e3 = [] # Store parameter counts including biases.
for idx_e3 in range(2): # Evaluate LeNet then AlexNet.
    H_e3, W_e3, C_e3 = inputs_e3[idx_e3] # Read one input shape.
    K_e3, S_e3, F_e3 = kernels_e3[idx_e3], strides_e3[idx_e3], filters_e3[idx_e3] # Read layer hyperparameters.
    spatial_e3.append(((H_e3 - K_e3) // S_e3 + 1, (W_e3 - K_e3) // S_e3 + 1)) # Compute valid output size.
    params_e3.append(K_e3 * K_e3 * C_e3 * F_e3 + F_e3) # Count weights plus biases.
print("spatial outputs:", spatial_e3) # Inspect output grids.
print("parameters:", params_e3) # Inspect trainable counts.
assert spatial_e3 == [(28, 28), (55, 55)] # Verify canonical spatial sizes.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a comparison chart.
plt.bar(names_e3, params_e3, color=["teal", "crimson"]) # Compare parameter counts.
plt.yscale("log") # Use log scale because AlexNet is much larger.
plt.title("Easy 3: first-layer parameter scale") # Title the chart.
plt.ylabel("parameters") # Label the scale.
plt.show() # Display the comparison.

▶ What you'll see: AlexNet conv1 has far more parameters even before considering the much larger activation tensor.

👀 Takeaway: AlexNet is not just deeper; it changes scale, width, stride, and channel count.

### Easy 4 — Visualize stride losing landing positions

**Goal.** Compare stride 1 and stride 4 landing grids, because AlexNet's first layer saves compute by evaluating fewer positions. We build it in 3 steps.

In [ ]:
H_e4, K_e4 = 15, 3 # Use a small one-dimensional analogy for readability.
starts_s1_e4 = np.arange(0, H_e4 - K_e4 + 1, 1) # All valid starts with stride 1.
starts_s4_e4 = np.arange(0, H_e4 - K_e4 + 1, 4) # Coarse starts with stride 4.
print("stride-1 starts:", starts_s1_e4) # Inspect dense landings.
print("stride-4 starts:", starts_s4_e4) # Inspect sparse landings.

▶ What you'll see: stride 4 keeps only a subset of the kernel starts.

In [ ]:
count_s1_e4 = len(starts_s1_e4) # Count stride-1 positions.
count_s4_e4 = len(starts_s4_e4) # Count stride-4 positions.
print("landing counts:", count_s1_e4, count_s4_e4) # Inspect compute reduction.
assert count_s1_e4 == 13 and count_s4_e4 == 4 # Verify the landing counts.

In [ ]:
plt.figure(figsize=(5, 2.8)) # Create a landing-position plot.
plt.scatter(starts_s1_e4, np.ones_like(starts_s1_e4), label="stride 1", color="gray") # Plot dense starts.
plt.scatter(starts_s4_e4, np.zeros_like(starts_s4_e4), label="stride 4", color="crimson") # Plot sparse starts.
plt.yticks([0, 1], ["stride 4", "stride 1"]) # Label rows.
plt.title("Easy 4: stride controls how often kernels land") # Title the plot.
plt.xlabel("input start coordinate"); plt.legend() # Label and legend.
plt.show() # Display the comparison.

▶ What you'll see: stride 4 skips many positions that stride 1 would inspect.

👀 Takeaway: large stride reduces activation size and compute, but it can miss small spatial details.

### Easy 5 — Estimate dense-classifier risk

**Goal.** Compare convolutional and dense parameter counts, because AlexNet needed dropout partly due to a huge fully connected classifier. We build it in 3 steps.

In [ ]:
conv_params_e5 = 3 * 3 * 256 * 256 + 256 # Count one 3×3 conv with 256 input and 256 output channels.
dense_params_e5 = 6 * 6 * 256 * 4096 + 4096 # Count AlexNet-style dense layer from 6×6×256 to 4096.
print("conv params:", conv_params_e5) # Inspect convolutional count.
print("dense params:", dense_params_e5) # Inspect dense count.
assert dense_params_e5 == 37752832 # Verify the cited dense count.

▶ What you'll see: the dense layer is orders of magnitude larger than the convolution.

In [ ]:
ratio_e5 = dense_params_e5 / conv_params_e5 # Compute how many times larger the dense layer is.
print("dense/conv ratio:", round(ratio_e5, 1)) # Inspect the parameter-budget ratio.
assert round(ratio_e5, 1) == 64.0 # Verify the rounded ratio for this comparison.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a log-scale budget plot.
plt.bar(["3×3 conv", "dense"], [conv_params_e5, dense_params_e5], color=["teal", "red"]) # Compare counts.
plt.yscale("log") # Use log scale so both are visible.
plt.title("Easy 5: dense classifier dominates") # Title the plot.
plt.ylabel("parameters") # Label the scale.
plt.show() # Display the chart.

▶ What you'll see: the dense layer dominates the budget even after spatial maps have shrunk.

👀 Takeaway: flattening removes spatial sharing, so dense classifiers need regularization and enough data.

## 🔴 Advanced

### Advanced 1 — Train a tiny convolutional detector

**Goal.** Learn one 3×3 filter with finite-difference gradients, because CNN filters are not hand-coded in practice; training discovers templates that reduce loss. We build it in 5 steps.

In [ ]:
patches_a1 = np.array([[[0, 1, 0], [0, 1, 0], [0, 1, 0]], [[1, 1, 1], [0, 0, 0], [0, 0, 0]], [[0, 0, 0], [1, 1, 1], [0, 0, 0]], [[0, 0, 1], [0, 1, 0], [1, 0, 0]]], dtype=float) # Four 3×3 patches.
y_a1 = np.array([1.0, 0.0, 0.0, 0.0]) # Only the vertical stroke is positive.
w_a1 = 0.1 * np.random.randn(3, 3) # Initialize one learnable filter.
print("initial score for positive patch:", round(float(np.sum(patches_a1[0] * w_a1)), 3)) # Inspect the starting detector score.

▶ What you'll see: the random filter starts with an arbitrary score on the positive vertical stroke.

In [ ]:
def sigmoid_a1(x_a1): # Define a small logistic squashing function.
    return 1 / (1 + np.exp(-x_a1)) # Return probabilities from scores.

def loss_a1(w_local_a1): # Define mean squared error for the current filter.
    scores_a1 = np.array([np.sum(p_a1 * w_local_a1) for p_a1 in patches_a1]) # Score every patch.
    preds_a1 = sigmoid_a1(scores_a1) # Convert scores to probabilities.
    return float(np.mean((preds_a1 - y_a1) ** 2)) # Return MSE.
print("initial loss:", round(loss_a1(w_a1), 4)) # Inspect the objective before training.

In [ ]:
for step_a1 in range(80): # Run a tiny finite-difference training loop.
    grad_a1 = np.zeros_like(w_a1) # Allocate a numeric gradient.
    eps_a1 = 1e-4 # Use a small finite-difference step.
    for r_a1 in range(3): # Loop over filter rows.
        for c_a1 in range(3): # Loop over filter columns.
            bump_a1 = np.zeros_like(w_a1) # Prepare a one-weight perturbation.
            bump_a1[r_a1, c_a1] = eps_a1 # Perturb this weight.
            grad_a1[r_a1, c_a1] = (loss_a1(w_a1 + bump_a1) - loss_a1(w_a1 - bump_a1)) / (2 * eps_a1) # Central-difference gradient.
    w_a1 -= 0.8 * grad_a1 # Gradient descent update.
print("final loss:", round(loss_a1(w_a1), 4)) # Inspect the objective after training.
assert loss_a1(w_a1) < 0.18 # Verify learning improved the detector.

In [ ]:
scores_a1 = np.array([np.sum(p_a1 * w_a1) for p_a1 in patches_a1]) # Score all patches after training.
preds_a1 = sigmoid_a1(scores_a1) # Convert scores to probabilities.
print("predictions:", np.round(preds_a1, 3)) # Inspect class probabilities.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a learned-filter heatmap.
plt.imshow(w_a1, cmap="coolwarm") # Visualize learned positive and negative weights.
plt.colorbar(label="weight") # Add a weight scale.
plt.title("Advanced 1: learned 3×3 detector") # Title the plot.
plt.show() # Display the filter.

▶ What you'll see: the learned filter gives the vertical-stroke patch the largest probability, and its weights form a vertical preference pattern.

👀 Takeaway: CNN filters are optimized templates, not fixed image-processing rules.

### Advanced 2 — Compare sigmoid and ReLU gradient flow

**Goal.** Inspect activation slopes, because AlexNet's ReLU helped deeper visual networks train faster than saturating nonlinearities. We build it in 4 steps.

In [ ]:
z_a2 = np.linspace(-8, 8, 401) # Create a wide pre-activation range.
sig_a2 = 1 / (1 + np.exp(-z_a2)) # Compute sigmoid activations.
relu_a2 = np.maximum(0, z_a2) # Compute ReLU activations.
print("sigmoid endpoints:", round(sig_a2[0], 3), round(sig_a2[-1], 3)) # Inspect saturation.

▶ What you'll see: sigmoid is near 0 at -8 and near 1 at +8.

In [ ]:
sig_grad_a2 = sig_a2 * (1 - sig_a2) # Sigmoid derivative.
relu_grad_a2 = (z_a2 > 0).astype(float) # ReLU derivative away from zero.
print("max sigmoid slope:", round(float(sig_grad_a2.max()), 3)) # Inspect the largest sigmoid derivative.
print("ReLU positive slope:", relu_grad_a2[-1]) # Inspect ReLU's positive-side slope.
assert round(float(sig_grad_a2.max()), 3) == 0.25 # Verify sigmoid's maximum derivative.

In [ ]:
small_sig_area_a2 = float(np.mean(sig_grad_a2 < 0.05)) # Measure how often sigmoid slopes are tiny on this range.
print("fraction of sigmoid slopes < 0.05:", round(small_sig_area_a2, 3)) # Inspect saturation prevalence.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a derivative comparison plot.
plt.plot(z_a2, sig_grad_a2, label="sigmoid slope") # Plot sigmoid derivative.
plt.plot(z_a2, relu_grad_a2, label="ReLU slope", linestyle="--") # Plot ReLU derivative.
plt.title("Advanced 2: activation slopes") # Title the plot.
plt.xlabel("pre-activation"); plt.ylabel("derivative"); plt.legend() # Label axes and legend.
plt.show() # Display the slope comparison.

▶ What you'll see: sigmoid slopes vanish at large magnitudes, while ReLU keeps slope 1 for positive pre-activations.

👀 Takeaway: ReLU helped AlexNet maintain usable gradients through a larger vision network.

### Advanced 3 — Quantify dropout's ensemble-like scaling

**Goal.** Simulate dropout on a dense feature vector, because AlexNet used dropout to reduce co-adaptation in its enormous fully connected layers. We build it in 4 steps.

In [ ]:
features_a3 = np.array([2.0, 1.0, 0.5, 3.0]) # Define four dense-layer activations.
keep_prob_a3 = 0.5 # Keep half the units on average.
weight_a3 = np.array([0.4, -0.2, 0.8, 0.1]) # Define a downstream linear classifier.
base_score_a3 = float(features_a3 @ weight_a3) # Score without dropout.
print("base score:", round(base_score_a3, 3)) # Inspect the no-dropout score.

▶ What you'll see: one dense score before applying any random dropout mask.

In [ ]:
rng_a3 = np.random.default_rng(3) # Create reproducible dropout masks.
scores_a3 = [] # Store stochastic scores.
for trial_a3 in range(2000): # Sample many dropout masks.
    mask_a3 = (rng_a3.random(features_a3.shape) < keep_prob_a3).astype(float) # Keep/drop each activation.
    dropped_a3 = features_a3 * mask_a3 / keep_prob_a3 # Inverted dropout rescales kept units.
    scores_a3.append(float(dropped_a3 @ weight_a3)) # Score the masked vector.
scores_a3 = np.array(scores_a3) # Convert scores to an array.
print("mean dropout score:", round(float(scores_a3.mean()), 3)) # Inspect expected score.
assert abs(scores_a3.mean() - base_score_a3) < 0.08 # Verify inverted dropout preserves the expected scale.

In [ ]:
print("score std from random masks:", round(float(scores_a3.std()), 3)) # Inspect how much stochasticity dropout injects.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a histogram of dropout scores.
plt.hist(scores_a3, bins=25, color="teal", alpha=0.8) # Show the distribution over masks.
plt.axvline(base_score_a3, color="red", linestyle="--", label="no-dropout score") # Mark the deterministic score.
plt.title("Advanced 3: dropout keeps expected scale") # Title the plot.
plt.xlabel("linear score"); plt.legend() # Label and legend.
plt.show() # Display the histogram.

▶ What you'll see: individual masks change the score, but the average stays close to the no-dropout score.

👀 Takeaway: dropout regularizes large dense layers by forcing predictions to survive missing activations.

### Advanced 4 — Show why data augmentation helps translation

**Goal.** Train a simple template on one position and test shifted strokes, because AlexNet relied on augmentation to make large models less sensitive to exact image placement. We build it in 4 steps.

In [ ]:
def make_stroke_a4(col_a4): # Create an 8×8 image with a vertical stroke at a chosen column.
    im_a4 = np.zeros((8, 8)) # Start with a blank image.
    im_a4[2:6, col_a4:col_a4+2] = 1.0 # Draw a 2-pixel-wide vertical stroke.
    return im_a4 # Return the image.

template_a4 = make_stroke_a4(3) # A template trained only at center position.
shifted_a4 = [make_stroke_a4(c_a4) for c_a4 in [1, 2, 3, 4, 5]] # Create shifted versions.
print("number of shifted examples:", len(shifted_a4)) # Inspect the test set size.

▶ What you'll see: five images contain the same stroke at different horizontal positions.

In [ ]:
flat_template_a4 = template_a4.reshape(-1) # Flatten the centered template for a naive dense matcher.
dense_scores_a4 = np.array([flat_template_a4 @ im_a4.reshape(-1) for im_a4 in shifted_a4]) # Score shifts by exact pixel overlap.
print("dense template scores:", dense_scores_a4) # Inspect position sensitivity.

In [ ]:
kernel_a4 = np.ones((4, 2)) # A local vertical-stroke kernel.
conv_scores_a4 = [] # Store the best convolutional response for each shifted image.
for im_a4 in shifted_a4: # Loop over shifted strokes.
    responses_a4 = [] # Store local responses inside this image.
    for r_a4 in range(8 - 4 + 1): # Valid row starts.
        for c_a4 in range(8 - 2 + 1): # Valid column starts.
            responses_a4.append(np.sum(im_a4[r_a4:r_a4+4, c_a4:c_a4+2] * kernel_a4)) # Local template response.
    conv_scores_a4.append(max(responses_a4)) # Translation-tolerant best response.
conv_scores_a4 = np.array(conv_scores_a4) # Convert to array.
print("best conv scores:", conv_scores_a4) # Inspect shift robustness.
assert np.all(conv_scores_a4 == 8) # Verify the local detector finds every shifted stroke.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a score comparison plot.
plt.plot([1, 2, 3, 4, 5], dense_scores_a4, marker="o", label="flattened template") # Plot dense overlap scores.
plt.plot([1, 2, 3, 4, 5], conv_scores_a4, marker="s", label="best conv response") # Plot convolution scores.
plt.title("Advanced 4: local filters handle shifts better") # Title the plot.
plt.xlabel("stroke column"); plt.ylabel("score"); plt.legend() # Label axes and legend.
plt.show() # Display the comparison.

▶ What you'll see: the flattened template scores highest only at the trained position, while the convolutional detector finds the stroke anywhere.

👀 Takeaway: convolution gives a translation-aware mechanism, and augmentation teaches the classifier to expect such shifts.

### Advanced 5 — Budget a small architecture table

**Goal.** Build a layer-by-layer shape and parameter table, because CNN architecture diagrams are executable arithmetic. We build it in 5 steps.

In [ ]:
layers_a5 = [
    ("conv", 32, 32, 1, 5, 1, 0, 6),
    ("pool", 28, 28, 6, 2, 2, 0, 6),
    ("conv", 14, 14, 6, 5, 1, 0, 16),
    ("pool", 10, 10, 16, 2, 2, 0, 16),
] # Store layer type, input H, input W, input C, kernel, stride, pad, output C.
print("layers:", len(layers_a5)) # Inspect the table length.

▶ What you'll see: a compact LeNet-like sequence with two convolutions and two pools.

In [ ]:
shapes_a5 = [] # Store output shapes.
params_a5 = [] # Store trainable parameters.
for kind_a5, H_a5, W_a5, C_a5, K_a5, S_a5, P_a5, F_a5 in layers_a5: # Loop over layers.
    Hout_a5 = (H_a5 + 2 * P_a5 - K_a5) // S_a5 + 1 # Compute output height.
    Wout_a5 = (W_a5 + 2 * P_a5 - K_a5) // S_a5 + 1 # Compute output width.
    shapes_a5.append((Hout_a5, Wout_a5, F_a5)) # Store output tensor shape.
    params_a5.append(K_a5 * K_a5 * C_a5 * F_a5 + F_a5 if kind_a5 == "conv" else 0) # Count conv params only.
print("output shapes:", shapes_a5) # Inspect the architecture outputs.
print("params:", params_a5) # Inspect trainable counts.
assert shapes_a5[-1] == (5, 5, 16) # Verify final tensor shape.

In [ ]:
flat_a5 = np.prod(shapes_a5[-1]) # Flatten final spatial tensor.
dense_units_a5 = 120 # Use a LeNet-style first dense layer size.
dense_params_a5 = flat_a5 * dense_units_a5 + dense_units_a5 # Count dense weights plus biases.
total_params_a5 = int(np.sum(params_a5) + dense_params_a5) # Compute architecture total through this dense layer.
print("flattened features:", flat_a5) # Inspect dense input width.
print("dense params:", dense_params_a5, "total through dense:", total_params_a5) # Inspect total budget.
assert flat_a5 == 400 # Verify 5*5*16.

In [ ]:
names_a5 = ["C1", "S2", "C3", "S4", "dense"] # Label layers for plotting.
param_plot_a5 = params_a5 + [dense_params_a5] # Combine conv/pool/dense parameter counts.
print("parameter table:", list(zip(names_a5, param_plot_a5))) # Inspect the final table.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a parameter budget plot.
plt.bar(names_a5, np.array(param_plot_a5) + 1, color=["teal", "gray", "teal", "gray", "crimson"]) # Add 1 so zero-parameter pools show on log scale.
plt.yscale("log") # Use log scale to show small and large counts together.
plt.title("Advanced 5: executable architecture budget") # Title the plot.
plt.ylabel("parameters + 1") # Label the adjusted scale.
plt.show() # Display the budget.

▶ What you'll see: pooling layers have no trainable parameters, convolutions are modest, and the dense layer becomes the largest block in this small network.

👀 Takeaway: shape and parameter tables reveal where computation, memory, and overfitting risk enter a CNN.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

LeNet and AlexNet show how convolutional architecture choices turn image sizes into feature sizes, parameter counts, and training behavior.

LeNet made small convolution and pooling stacks practical for digit recognition. AlexNet scaled the same bookkeeping to much larger color images with ReLUs and large dense heads. The arithmetic of sizes, weights, and activations is the safest way to adapt these historic designs without copying them blindly.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)


## The concept, built once (D1)
$$H_{out}=\left\lfloor\frac{H-K+2P}{S}\right\rfloor+1,\qquad \text{weights}=K_hK_wC_{in}C_{out}$$

We first write the reusable method and assert the exact lesson numbers before scaling to the ladder.

In [ ]:

def conv_output_size(size, kernel, padding=0, stride=1):
    return int(np.floor((size - kernel + 2 * padding) / stride) + 1)


def tiny_lenet():
    conv_h = conv_output_size(32, 5)
    conv_w = conv_output_size(32, 5)
    conv_channels = 6
    conv_weights = 5 * 5 * 1 * conv_channels
    pool_shape = (conv_h // 2, conv_w // 2, conv_channels)
    dense_params = 6 * 6 * 256 * 4096 + 4096
    relu_demo = np.maximum(np.array([-3.0, 0.5, 4.0]), 0.0)
    return (conv_h, conv_w, conv_channels), conv_weights, pool_shape, dense_params, relu_demo


conv_shape, conv_weights, pool_shape, dense_params, relu_demo = tiny_lenet()
alex_first_shape = (55, 55, 96)
alex_first_weights = 11 * 11 * 3 * 96

assert conv_shape == (28, 28, 6)
assert conv_weights == 150
assert pool_shape == (14, 14, 6)
assert dense_params == 37752832
assert alex_first_shape == (55, 55, 96)
assert alex_first_weights == 34848
assert np.allclose(relu_demo, [0.0, 0.5, 4.0])

print("LeNet conv shape", conv_shape)
print("LeNet conv weights", conv_weights)
print("LeNet pool shape", pool_shape)
print("AlexNet first conv weights", alex_first_weights)
print("dense parameters", dense_params)


## Visual check
The numbers above are easier to trust when the intermediate feature behavior is visible.

In [ ]:

stages = ["input", "conv", "pool", "dense"]
heights = [32, 28, 14, 6]
channels = [1, 6, 6, 256]

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

axes[0].plot(stages, heights, marker="o")
axes[0].set_ylabel("spatial width")
axes[0].set_title("LeNet spatial sizes")

axes[1].bar(stages, channels)
axes[1].set_ylabel("channels")
axes[1].set_title("channel plan")

plt.tight_layout()
plt.show()


## Dataset ladder (D1 to D5)
We inline the shared CPU-safe classification ladder. Each rung returns images `X` with shape `(n, 8, 8)` and labels `y`, so the same featurizer can be evaluated from hand patches to the hardest fallback or cached MNIST rung.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab),
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1], a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety), the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable, else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift), else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)




rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray")
    ax.set_title(f"{name.split()[0]}\n{X.shape}\n{len(set(y.tolist()))} classes")
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout()
plt.show()

for name, X, y in rungs:
    print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))}")


## Run the same method across D1-D5
Only the data rung changes. The featurizer and accuracy metric stay fixed.

In [ ]:

def max_pool_2x2(img):
    pooled = img.reshape(4, 2, 4, 2).max(axis=(1, 3))
    return pooled


def topic_feature_map(img):
    relu = np.maximum(img - img.mean(), 0.0)
    pooled = max_pool_2x2(relu)
    return pooled


def featurize(img):
    pooled = topic_feature_map(img)
    return np.concatenate([img.ravel(), pooled.ravel()])


accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(featurize, X, y)
    accuracies.append(acc)
    print(f"{name:38s} accuracy={acc:.3f}")

baseline_accuracies = []

for name, X, y in rungs:
    acc = accuracy_with(lambda im: im.ravel(), X, y)
    baseline_accuracies.append(acc)

print("flat baseline", [round(x, 3) for x in baseline_accuracies])


## Results visualization
Top row: one feature or activation panel per rung. Bottom row: accuracy versus ladder complexity.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for idx, (name, X, y) in enumerate(rungs):
    feature = topic_feature_map(X[0])
    if isinstance(feature, tuple):
        feature = feature[0]
    axes[0, idx].imshow(feature, cmap="viridis")
    axes[0, idx].set_title(name.split()[0])
    axes[0, idx].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

axes[1, 0].plot(range(1, 6), accuracies, marker="o", label="topic features")
axes[1, 0].plot(range(1, 6), baseline_accuracies, marker="s", label="flat baseline")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_ylim(0.0, 1.05)
axes[1, 0].legend()
axes[1, 0].set_title("accuracy vs rung")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: flattening or striding too early
AlexNet used a large first kernel for a large input, but copying that idea to tiny images destroys detail. We reproduce an over-aggressive stride summary on D5, then keep more signal with a smaller ReLU-plus-pool stage.

In [ ]:

sample = rungs[-1][1][0]
large_kernel_summary = sample[::4, ::4]
small_kernel_summary = max_pool_2x2(np.maximum(sample - sample.mean(), 0.0))
wrong_detail = large_kernel_summary.var()
fixed_detail = small_kernel_summary.var()

print("blind stride-4 sample shape", large_kernel_summary.shape)
print("small-kernel pooled shape", small_kernel_summary.shape)
print("detail variance lost", np.round(wrong_detail, 4))
print("detail variance kept", np.round(fixed_detail, 4))


## Evaluate it + Practice
- Metric: held-out accuracy on every rung, compared with a no-skill flat-pixel logistic baseline.
- Sanity check: D1 should be easy enough to overfit or nearly overfit with the concept features.
- Ablation: remove the key block feature and accuracy should not improve over the flat baseline.
- Failure signal: the hardest rung may expose distribution shift, shape mismatch, or compute-budget mistakes before D1 does.

Practice prompts:
1. Change one design constant and rerun the accuracy curve.
2. Print the D5 confusion pattern for the worst two classes.
3. Replace the feature map panel with an example from a different class.

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here
